# Understanding DeepSeek: from token IDs to vectors

This notebook introduces the first building block of a DeepSeek-style language model: **token embeddings**. A tokenizer represents text as integer IDs; the embedding layer maps each ID to a learned vector that later Transformer blocks can process.

We will track three tensor dimensions throughout:

- `batch`: how many sequences are processed together.
- `sequence`: how many token positions each sequence contains.
- `dim`: the width of each token representation.

The numbers here are deliberately small so the shapes and parameter counts are easy to inspect.

## 1. Model configuration

`DeepSeekConfig` stores the dimensions shared by the model. The fields most relevant to this notebook are:

| Setting | Meaning | Value |
| --- | --- | ---: |
| `vocab_size` | Number of token IDs | 8,192 |
| `max_seq_len` | Maximum sequence length | 1,024 |
| `dim` | Width of each token vector | 256 |
| `n_layers` | Planned Transformer blocks | 6 |
| `n_heads` | Attention heads per block | 8 |
| `n_routed_experts` | Routed MoE experts | 8 |
| `n_activated_experts` | Experts selected per token | 2 |

The MLA and MoE settings are introduced now so later notebooks can reuse the same configuration object.

In [1]:

from dataclasses import dataclass

@dataclass
class DeepSeekConfig:
    # Tokenization and sequence length
    vocab_size: int = 8192
    max_seq_len: int = 1024

    # Transformer architecture
    dim: int = 256
    n_layers: int = 6
    n_dense_layers: int = 1
    n_heads: int = 8

    # Dense and expert feed-forward networks
    inter_dim: int = 768
    moe_inter_dim: int = 256

    # Multi-Head Latent Attention
    q_lora_rank: int = 128
    kv_lora_rank: int = 64
    qk_nope_head_dim: int = 32
    qk_rope_head_dim: int = 16
    v_head_dim: int = 32

    # Mixture of Experts
    n_routed_experts: int = 8
    n_shared_experts: int = 1
    n_activated_experts: int = 2

    # Routing
    n_expert_groups: int = 1
    n_limited_groups: int = 1
    route_scale: float = 1.0

    # Positional encoding
    rope_theta: float = 10000.0

    def __post_init__(self):
        assert self.dim > 0
        assert self.n_heads > 0
        assert 0 <= self.n_dense_layers <= self.n_layers
        assert 1 <= self.n_activated_experts <= self.n_routed_experts
        assert self.qk_rope_head_dim % 2 == 0
        assert self.kv_lora_rank > 0
        assert self.q_lora_rank > 0

config = DeepSeekConfig()
print(config)


DeepSeekConfig(vocab_size=8192, max_seq_len=1024, dim=256, n_layers=6, n_dense_layers=1, n_heads=8, inter_dim=768, moe_inter_dim=256, q_lora_rank=128, kv_lora_rank=64, qk_nope_head_dim=32, qk_rope_head_dim=16, v_head_dim=32, n_routed_experts=8, n_shared_experts=1, n_activated_experts=2, n_expert_groups=1, n_limited_groups=1, route_scale=1.0, rope_theta=10000.0)


In [2]:
import torch

config = DeepSeekConfig()

## 2. Implement the embedding lookup

The embedding table has shape `[vocab_size, dim]`, which is `[8,192, 256]` in this configuration. Each integer token ID selects one row from that table.

`torch.nn.Embedding` performs this lookup efficiently. It does not add context or perform attention; it only provides the initial learned vector for each token.

In [3]:
import torch
import torch.nn as nn

config = DeepSeekConfig()

class TokenEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.embedding = nn.Embedding(
            config.vocab_size,
            config.dim
        )

    def forward(self, token_ids):
        return self.embedding(token_ids)

In [5]:

model = TokenEmbedding(config)

# Two sequences, each containing five token IDs. The IDs are illustrative;
# a real tokenizer would decide how text is split into token pieces.
tokens = torch.tensor([
    [15, 82, 7, 104, 21],
    [11, 24, 35, 45, 19]
])

output = model(tokens)

print("Input shape:", tokens.shape)
print("Output shape:", output.shape)
print("Parameter count:", sum(
    p.numel() for p in model.parameters()
))

Input shape: torch.Size([2, 5])
Output shape: torch.Size([2, 5, 256])
Parameter count: 2097152


## 3. Interpret the output shape

The embedding preserves the batch and sequence axes, then appends one vector of length `config.dim` for every token:

```text
[batch, sequence] → [batch, sequence, dim]
[2, 5]            → [2, 5, 256]
```

The table contains `8,192 × 256 = 2,097,152` trainable values. The next cell turns this shape contract into an executable check.

In [ ]:
assert tokens.shape == (2, 5)
assert output.shape == (2, 5, config.dim)
assert sum(p.numel() for p in model.parameters()) == config.vocab_size * config.dim
print("Shape contract verified.")